In [ ]:
!pip install transformers

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from tqdm import tqdm
from torch.utils.data import DataLoader

# GPU 설정
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("device: ", device)

In [ ]:
df = pd.read_excel('/content/sample_data/Ai-hub_선정적&비선정적.xlsx')
df = df.drop_duplicates(subset=['origin_text'])
df = df[['origin_text', 'is_sexual']]
df['label'] = df['is_sexual']
df = df[['origin_text', 'label']]
df.dropna(inplace=True)

In [ ]:
train_data = df.sample(frac=0.8, random_state=42)
test_data = df.drop(train_data.index)

In [ ]:
MODEL_NAME = "beomi/KcELECTRA-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# 토큰화
tokenized_train_sentences = tokenizer(
    list(train_data['origin_text']),
    return_tensors="pt",
    max_length=128,
    padding=True,
    truncation=True,
    add_special_tokens=True
)

In [ ]:
tokenized_test_sentences = tokenizer(
    list(test_data['origin_text']),
    return_tensors="pt",
    max_length=128,
    padding=True,
    truncation=True,
    add_special_tokens=True
)

In [ ]:
# 데이터셋 클래스 정의
class CurseDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_label = train_data["label"].values
test_label = test_data["label"].values

train_dataset = CurseDataset(tokenized_train_sentences, train_label)
test_dataset = CurseDataset(tokenized_test_sentences, test_label)

In [ ]:
# 데이터 로더
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# 모델 정의 (dropout 추가)
class CustomModel(AutoModelForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.dropout = torch.nn.Dropout(0.5)  # dropout 비율 설정

model = CustomModel.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

In [ ]:
from transformers import EarlyStoppingCallback, TrainingArguments, Trainer

# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir='./',
    num_train_epochs=5,
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_steps=500,
    save_steps=500,  # save_steps를 eval_steps와 일치시킴
    eval_steps=500,   # eval_steps 추가
    save_total_limit=10,
    load_best_model_at_end=True,
    metric_for_best_model="pr-auc",
    greater_is_better=True,
)

In [ ]:
# EarlyStoppingCallback 설정
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=2)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, roc_auc_score, precision_recall_curve, auc

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)  # 예측된 클래스 (0 또는 1)
    preds_prob = pred.predictions[:, 1]  # 긍정 클래스(1)에 대한 확률값

    # Precision, Recall, F1, Accuracy 계산
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    # AUC-ROC 계산
    auc_roc = roc_auc_score(labels, preds_prob)

    # PR-AUC 계산
    precision_vals, recall_vals, _ = precision_recall_curve(labels, preds_prob)
    pr_auc = auc(recall_vals, precision_vals)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc-roc': auc_roc,
        'pr-auc': pr_auc
    }

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# 클래스 가중치 계산
class_weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=train_label)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

# 커스텀 Trainer 클래스 정의
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        # forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # 손실 함수에 클래스 가중치 적용
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# CustomTrainer 설정
trainer = CustomTrainer(
    model=model,  # 모델 객체
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping_callback]  # 콜백에 추가
)

In [ ]:
# 학습 시작
trainer.train()

In [ ]:
print("Final Test Results:")
test_results = trainer.evaluate(eval_dataset=test_dataset)
trainer.log_metrics("test", test_results)
trainer.save_metrics("test", test_results)

# 최종 결과: 훈련 데이터셋에 대한 평가
print("Final Train Results:")
train_results = trainer.evaluate(eval_dataset=train_dataset)
trainer.log_metrics("train", train_results)
trainer.save_metrics("train", train_results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def make_contiguous(model):
    for param in model.parameters():
        param.data = param.data.contiguous()

# 모델의 파라미터를 연속적으로 변환
make_contiguous(model)

# 모델 저장 경로 설정
model_save_path = '/content/drive/MyDrive/2024/saved_model/KcELECTRA_Ver2'  # 원하는 경로로 변경

# 모델 저장
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

# 제목 수치화

In [ ]:
import torch.nn.functional as F
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_save_path = '/content/drive/MyDrive/2024/saved_model/KcELECTRA_Ver2'

# 1. 모델과 토크나이저 불러오기
model = AutoModelForSequenceClassification.from_pretrained(model_save_path)
tokenizer = AutoTokenizer.from_pretrained(model_save_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# 기사 제목의 선정성 확률을 계산하는 함수
def predict_title_sensationalism_with_score(title, model, tokenizer, device):
    model.eval()

    # 입력된 기사 제목을 토크나이즈
    inputs = tokenizer(
        title,
        return_tensors="pt",
        truncation=True,
        add_special_tokens=True,
        max_length=128
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # 모델에 입력하고 예측 결과 및 logits 얻기
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Softmax를 통해 확률로 변환
        probs = F.softmax(logits, dim=1)
        sensational_prob = probs[0][1].item()  # 선정적인 클래스(1)의 확률

    return sensational_prob

## 제목전처리X

In [ ]:
test_df = pd.read_excel('/content/sample_data/크롤링데이터_제목전처리_최종 (1).xlsx')

In [ ]:
test_df

In [ ]:
# '수치' 열을 추가
test_df['선정성 수치'] = 0.0

# 각 제목에 대해 선정성 확률 계산 후 '수치' 열에 추가
for index in range(len(test_df)):
    title = test_df['제목'].iloc[index]  # '새로운 제목' 열에서 제목을 가져옵니다.
    sensational_prob = predict_title_sensationalism_with_score(title, model, tokenizer, device)
    test_df.at[index, '선정성 수치'] = sensational_prob  # 확률 값을 '수치' 열에 추가

In [ ]:
sorted_df = test_df.sort_values(by='선정성 수치', ascending=False)
sorted_df

In [ ]:
sorted_df = sorted_df.head(100)

In [ ]:
sorted_df.info()

In [ ]:
sorted_df.to_excel('KcELECTRA_제목전처리O_수치화_전체.xlsx', index=False)

## 제목전처리

In [ ]:
test_df = pd.read_excel('/content/sample_data/크롤링데이터_제목전처리_최종.xlsx')

In [ ]:
# '수치' 열을 추가
test_df['수치'] = 0.0

# 각 제목에 대해 선정성 확률 계산 후 '수치' 열에 추가
for index in range(len(test_df)):
    title = test_df['제목'].iloc[index]  # '제목' 열에서 제목을 가져옵니다.
    sensational_prob = predict_title_sensationalism_with_score(title, model, tokenizer, device)
    test_df.at[index, '수치'] = sensational_prob  # 확률 값을 '수치' 열에 추가

In [ ]:
sorted_df = test_df.sort_values(by='수치', ascending=False)
sorted_df

In [ ]:
sorted_df = sorted_df.head(100)

In [ ]:
sorted_df.to_excel('제목전처리O_수치화.xlsx', index=False)